# Single-sample Gemini Audio Inference (Arabic)

Runs `gemini-2.5-flash` for the missing Arabic sample in `uh-mazing_gemini-2.5-flash_audio_standard-noextra.csv` and prints the output.

In [ ]:
import os
from pathlib import Path

import google.generativeai as genai
import pandas as pd
from dotenv import load_dotenv


In [ ]:
load_dotenv()

MODEL_NAME = 'gemini-2.5-flash'
INPUT_CSV = Path('../outputs/uh-mazing_gemini-2.5-flash_audio_standard-noextra.csv')
AUDIO_DIR = Path('../asr/preprocessed_audio_switchboard')
ROW_INDEX = 65
TARGET_LANG = 'Arabic'
PROMPT = (
    'Translate the speech audio into <LANGUAGE> text. '
    'Only return the answer requested. Do not include any explanation or introductions.'
).replace('<LANGUAGE>', TARGET_LANG)

api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
if not api_key:
    raise RuntimeError('Missing GEMINI_API_KEY (or GOOGLE_API_KEY)')

genai.configure(api_key=api_key)
model = genai.GenerativeModel(MODEL_NAME)

df = pd.read_csv(INPUT_CSV)
row = df.loc[ROW_INDEX]
audio_name = str(row['_audio_filename'])
audio_path = AUDIO_DIR / audio_name

print('Row index:', ROW_INDEX)
print('ID:', row['ID'])
print('Audio file:', audio_path)
print('Current AR value:', row.get('AR_gemini-2.5-flash'))

if not audio_path.exists():
    raise FileNotFoundError(f'Audio not found: {audio_path}')


In [ ]:
audio_bytes = audio_path.read_bytes()
response = model.generate_content(
    [
        PROMPT,
        {"mime_type": "audio/wav", "data": audio_bytes},
    ],
    generation_config={"temperature": 0.0},
)

translation = (response.text or "").strip()
print("\nArabic translation:")
print(translation)
